In [1]:
#packages
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD, KNNBaseline, KNNBasic, NormalPredictor
from surprise.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel


In [2]:
ratings = pd.read_csv('ratings_clean.csv')
recipes = pd.read_csv('recipes_clean.csv')

#ratings.head()
reader = Reader(rating_scale=(1,5))
data = Dataset.load_from_df(
    ratings[['user_id', 'recipe_id', 'rating']],
    reader
)


In [3]:
svd = SVD()
trainset = data.build_full_trainset()
svd.fit(trainset)

In [4]:
tfidf = TfidfVectorizer(stop_words='english')
recipes['ingredients'] = recipes['ingredients'].fillna('')

In [5]:
#without ingredients
def recommender(userId):
    recipes_r = recipes.copy()
    recipes_r['est'] = recipes_r['id'].apply(lambda x: svd.predict(userId, x).est)

    recipes_r = recipes_r.sort_values('est', ascending=False)

    return recipes_r[['name', 'est']].head(5)


recommender(1701315) #101823 868654 1701315


,name,est
11,symphony brownies,5.0
12486,zurie s overnight no knead bread,5.0
12485,zuppa toscana soup olive garden clone,5.0
6,never weep whipped cream,5.0
14,wacky cake,5.0


In [6]:
def recommender_ingredients(userId, my_ingredients):
    all_ingredients = list(recipes['ingredients']) + [my_ingredients]

    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(all_ingredients)

    cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

    my_index = len(recipes)

    sim_scores = list(enumerate(cosine_sim[my_index]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:26]

    recipe_indices = [i[0] for i in sim_scores]

    recipies_c = recipes.iloc[recipe_indices][['name', 'n_ingredients', 'id']]

    recipies_c['est'] = recipies_c['id'].apply(lambda x: svd.predict(userId, x).est)
    recipies_c = recipies_c.sort_values('est', ascending=False)

    return recipies_c.head(10)


In [7]:
recommender_ingredients(1701315, 'beef bread orange')

,name,n_ingredients,id,est
7787,moroccan orange cake,9,424183,5.000000
8320,orange cranberry sauce,5,73953,5.000000
3272,cranberry nut bread,11,42918,5.000000
4458,eggs with hats on top,3,54677,5.000000
6062,iced tea,8,61645,5.000000
3277,cranberry orange muffins,10,139499,5.000000
1318,best orange dreamsicle cake,8,109825,5.000000
10221,simple ranch house meatloaf,13,39864,4.997579
610,awesome orange chocolate muffins,11,440046,4.987639
2928,cinnamon toast deluxe,4,93128,4.969042
